# Plain RAG baseline

Goal: rebuild the docs RAG from the RAG module so there is something to compare the agent against.

## Load the API key

`find_dotenv` walks up from this folder to the repo root, so the notebook works at any depth.

In [ ]:
from dotenv import load_dotenv, find_dotenv

# walk up from this notebook's folder until a .env is found
env_path = find_dotenv(usecwd=True)
assert env_path, ".env not found in this folder or any parent"
load_dotenv(env_path)

## Smoke test the client

One cheap call to prove the key and client work before doing anything bigger.

In [ ]:
from openai import OpenAI

openai_client = OpenAI()

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Tell me a joke"
)

print(response.output_text)

## Load and chunk the Evidently docs

Same `gitsource` pipeline as before: read the md/mdx files, parse frontmatter, split into overlapping 3000-character chunks.

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

## Index the chunks

Keyword search over title, description and content, with `filename` kept as a filter field.

In [ ]:
from minsearch import AppendableIndex

In [ ]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

## Search function

This is the function the model will later be allowed to call, so its signature matters: one `query` string in, a list of chunks out.

In [ ]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

## RAG prompt

Classic RAG: we search, paste the results into the prompt, and the model only answers.

In [ ]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

## Ask the question

Note the typo in `dahsbord` - it goes straight into the search as-is.

In [ ]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

## Get the answer

In [ ]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [ ]:
print(response.output_text)

The baseline fails: keyword search on the raw, misspelled question did not surface the dashboard pages, so the model correctly says the context does not cover it.

# Give the model a search tool

Goal: let the model decide what to search for, and run one tool call by hand to see every piece of the round trip.

## Instructions and tool schema

The instructions no longer carry the context - the model is told a knowledge base exists. The schema is what it reads to know the tool's name and arguments.

In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

## What a tool costs in tokens

Same call twice, without and with `tools`, to compare input tokens.

In [ ]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

In [ ]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

63 to 110: the tool schema is sent as input on every call, about 47 tokens here. Small for one tool, but it adds up with many.

## The model asks for a search

In [ ]:
tool_call = response.output[0]
tool_call

No text answer this time - `output[0]` is a function call. Look at the query: the model rewrote the question into `"create dashboard in Evidently"` and fixed the typo. That rewrite is the whole reason the second answer works. `arguments` is a JSON string, not a dict, hence `json.loads` below.

## Record the call and run the search

The tool call goes into `messages` first, so on the next turn the model sees its own request. Then we parse the arguments and run `search` ourselves - the model never executes anything.

In [ ]:
messages.append(tool_call)

In [ ]:
tool_call.arguments

In [ ]:
arguments = json.loads(tool_call.arguments)
arguments

In [ ]:
search_results = search(query='create dashboard in Evidently')

In [ ]:
search_results = search(**arguments)
search_results[:1]

## Send the result back

The `call_id` ties this output to the request above. Without the match the API rejects the history.

In [ ]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [ ]:
messages.append(call_output)
messages

## Second call - now it can answer

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

Input jumps to about 3,700 tokens - that is the five search results now sitting in the history.

In [ ]:
print(response.output_text)

Same question, same index, and now a real answer. The wording and steps will differ between runs.